# Gold Layer Consolidation Notebook
## Leer todos los Parquet de datalake_silver/ y consolidarlos con PySpark

Este notebook reproduce la lógica del `gold_processing_dag.py` para consolidar datos de Silver en Gold usando PySpark.

In [ ]:
# Sección 1: Requisitos y configuración del entorno
import os
import glob
from datetime import datetime
from typing import Dict, List, Optional
import logging

import pandas as pd
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Definir rutas
SILVER_DIR = os.getenv('SILVER_DIR', '/opt/airflow/datalake_silver')
GOLD_DIR = os.getenv('GOLD_DIR', '/opt/airflow/datalake_gold')

if not os.path.exists(SILVER_DIR):
    SILVER_DIR = os.path.abspath('./datalake_silver')
    GOLD_DIR = os.path.abspath('./datalake_gold')

os.makedirs(GOLD_DIR, exist_ok=True)

print(f"SILVER_DIR: {SILVER_DIR} (existe: {os.path.exists(SILVER_DIR)})")
print(f"GOLD_DIR: {GOLD_DIR} (existe: {os.path.exists(GOLD_DIR)})")

In [ ]:
# Sección 2: Crear SparkSession
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("GoldConsolidation")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

print(f"SparkSession creada: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

In [ ]:
# Sección 3: Inspeccionar archivos en datalake_silver
parquet_files = sorted(glob.glob(os.path.join(SILVER_DIR, "*.parquet")))

print(f"Total de Parquet en {SILVER_DIR}: {len(parquet_files)}")
for fpath in parquet_files[:10]:
    fname = os.path.basename(fpath)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  - {fname} ({size_mb:.2f} MB)")

In [ ]:
# Sección 4: Clasificar archivos por familia
FAMILIES = {"silver_titles": [], "silver_rt_reviews": [], "silver_tmdb_reviews": [], "ignored": []}

for fpath in parquet_files:
    basename = os.path.basename(fpath)
    if basename.startswith("silver_titles"):
        FAMILIES["silver_titles"].append(fpath)
    elif basename.startswith("silver_rt_reviews"):
        FAMILIES["silver_rt_reviews"].append(fpath)
    elif basename.startswith("silver_tmdb_reviews"):
        FAMILIES["silver_tmdb_reviews"].append(fpath)
    else:
        FAMILIES["ignored"].append(fpath)

print("Clasificación:")
for family, files in FAMILIES.items():
    print(f"  {family}: {len(files)} archivos")

In [ ]:
# Sección 5: Leer archivos Parquet con PySpark
silver_dfs = {}

for family in ["silver_titles", "silver_rt_reviews", "silver_tmdb_reviews"]:
    files = FAMILIES.get(family, [])
    if files:
        try:
            df = spark.read.parquet(*files)
            count = df.count()
            cols = len(df.columns)
            silver_dfs[family] = df
            print(f"✓ {family}: {count:,} filas, {cols} columnas")
        except Exception as e:
            print(f"✗ Error leyendo {family}: {e}")

In [ ]:
# Mostrar esquemas
for family, df in silver_dfs.items():
    print(f"\nEsquema {family}:")
    df.printSchema()

In [ ]:
# Sección 7: Muestrear datos
for family, df in silver_dfs.items():
    print(f"\nMuestra de {family}:")
    df.limit(2).show(truncate=False)

In [ ]:
# Sección 8: Funciones de limpieza
from pyspark.sql.functions import col, lit, current_timestamp, row_number, isnan, isnull, when
from pyspark.sql import Window

def clean_and_enrich(df, family_name):
    """Limpiar, desduplicar y enriquecer datos."""
    
    original_count = df.count()
    
    # Opcionalmente desduplicar por columnas key
    if "canonical_title" in df.columns and "review_id" in df.columns:
        df = df.dropDuplicates(subset=["canonical_title", "review_id"])
    elif "canonical_title" in df.columns:
        df = df.dropDuplicates(subset=["canonical_title"])
    
    # Agregar metadatos
    df = df.withColumn("gold_processed_at", current_timestamp())
    df = df.withColumn("gold_family", lit(family_name))
    
    final_count = df.count()
    print(f"{family_name}: {original_count:,} -> {final_count:,} filas")
    
    return df

# Aplicar limpieza
cleaned_dfs = {}
for family, df in silver_dfs.items():
    cleaned_dfs[family] = clean_and_enrich(df, family)

print("\n✓ Limpieza completada")

In [ ]:
# Sección 9: Escribir archivos Gold
run_stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
print(f"Timestamp: {run_stamp}\n")

written_files = {}
for family, df in cleaned_dfs.items():
    filename = f"gold_{family}_{run_stamp}.parquet"
    output_path = os.path.join(GOLD_DIR, filename)
    
    try:
        df.coalesce(1).write.mode("overwrite").parquet(output_path)
        written_files[family] = output_path
        print(f"✓ Escrito: {filename}")
    except Exception as e:
        print(f"✗ Error escribiendo {family}: {e}")

print(f"\n✓ {len(written_files)} archivos Gold escritos")

In [ ]:
# Sección 10: Validar
gold_files = sorted(glob.glob(os.path.join(GOLD_DIR, "*.parquet")))

print(f"Archivos finales en Gold: {len(gold_files)}\n")
total_size_mb = 0
for fpath in gold_files:
    fname = os.path.basename(fpath)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    total_size_mb += size_mb
    print(f"  - {fname} ({size_mb:.2f} MB)")

print(f"\nTamaño total: {total_size_mb:.2f} MB")

In [ ]:
# Sección 11: Resumen
print("\n" + "="*60)
print("CONSOLIDACIÓN GOLD COMPLETADA")
print("="*60)
print(f"\nTimestamp: {run_stamp}")
print(f"\nFamilias procesadas:")
for family, df in cleaned_dfs.items():
    print(f"  - {family}: {df.count():,} filas")

print(f"\nArchivos generados en {GOLD_DIR}:")
for family, path in written_files.items():
    print(f"  - {os.path.basename(path)}")

print(f"\n✓ Listo para consumo por dashboards en Workshop 4")